# Descarga de datos del INE

Se utiliza la API pública del INE (`servicios.ine.es/wstempus`) para descargar cada tabla por su ID numérico y guardarla como `.csv`. Si el archivo ya existe en disco, se omite la descarga para evitar duplicados. Los datos se filtran al período **2005–2025**.
```

In [ ]:
"""import os
import requests
import pandas as pd
import datetime

CARPETA = "/Users/fran/Documents/Proyectos máster/TFM/"
os.makedirs(CARPETA, exist_ok=True)


def descargar(nombre_archivo, id_tabla, descripcion):

    ruta = CARPETA + nombre_archivo

    if os.path.exists(ruta):
        print(f"✅ Ya existe, omitiendo: {nombre_archivo}")
        return

    print(f"⬇️  Descargando: {descripcion}...")

    url = f"https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/{id_tabla}?tip=AM&lang=ES&nult=400"

    try:
        respuesta = requests.get(url, timeout=120)
        respuesta.raise_for_status()
        datos = respuesta.json()
    except Exception as e:
        print(f"ERROR tabla {id_tabla}: {e}\n")
        return

    # A veces el INE devuelve un string de error en vez de una lista
    if not isinstance(datos, list):
        print(f"Tabla {id_tabla} devolvió: {str(datos)[:200]}\n")
        return

    filas = []
    for serie in datos:
        if not isinstance(serie, dict):
            continue
        for obs in serie.get("Data", []):
            if obs.get("Fecha") is None:
                continue
            try:
                fecha = datetime.datetime.utcfromtimestamp(int(obs["Fecha"]) / 1000)
            except (ValueError, TypeError):
                fecha = pd.to_datetime(obs["Fecha"]).to_pydatetime()
            filas.append({
                "Serie":   serie.get("Nombre"),
                "Fecha":   fecha.strftime("%Y-%m-%d"),
                "Periodo": obs.get("NombrePeriodo"),
                "Valor":   obs.get("Valor"),
            })

    if not filas:
        print(f" Tabla {id_tabla} devolvió 0 observaciones. ID incorrecto.\n")
        return

    df = pd.DataFrame(filas)
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df["Fecha"].dt.year.between(2003, 2025)]

    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"   💾 Guardado: {nombre_archivo}  ({len(df):,} filas)\n")

descargar("03_edad_media_provincia.csv",   3199, "Edad media por provincia")
descargar("04_ipc_ccaa.csv",               76136, "IPC por CCAA (mensual)")
descargar("05_precio_mano_obra",           8377, "Indice precio mano de obra (mensual)") # No tengo la base
descargar("06_precio_materiales.csv",      8381,  " Indice Precio materiales de construcción")
descargar("07_compraventa_viviendas.csv",  6150, "Compraventa de viviendas por provincia") # Datos desde 2007
descargar("08_hipotecas.csv",              3200,  "Hipotecas concedidas por provincia")
descargar("09_poblacion_total.csv",        2852, 67988  "Población total en España")"""

'import os\nimport requests\nimport pandas as pd\nimport datetime\n\nCARPETA = "/Users/fran/Documents/Proyectos máster/TFM/"\nos.makedirs(CARPETA, exist_ok=True)\n\n\ndef descargar(nombre_archivo, id_tabla, descripcion):\n\n    ruta = CARPETA + nombre_archivo\n\n    if os.path.exists(ruta):\n        print(f"✅ Ya existe, omitiendo: {nombre_archivo}")\n        return\n\n    print(f"⬇️  Descargando: {descripcion}...")\n\n    url = f"https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/{id_tabla}?tip=AM&lang=ES&nult=400"\n\n    try:\n        respuesta = requests.get(url, timeout=120)\n        respuesta.raise_for_status()\n        datos = respuesta.json()\n    except Exception as e:\n        print(f"ERROR tabla {id_tabla}: {e}\n")\n        return\n\n    # A veces el INE devuelve un string de error en vez de una lista\n    if not isinstance(datos, list):\n        print(f"Tabla {id_tabla} devolvió: {str(datos)[:200]}\n")\n        return\n\n    filas = []\n    for serie in datos:\n        if no

# Importamos el dato `MIR.M.ES.B.A2C.A.C.A.2250.EUR.N` desde el BCE, ya que lo necesitamos para saber el tipo medio que ofrecen los bancos a los clientes.

In [101]:
pip install ecbdata

Note: you may need to restart the kernel to use updated packages.


Tras consultar en la página del BCE cuál es el dato más apropiado para el tipo medio de hipotecas concedidas por los bancos durante un periodo t, nos cpiamos el códio del dato y lo manipulamos con la librería `ecbdata`

In [102]:
from ecbdata import ecbdata

df = ecbdata.get_series('MIR.M.ES.B.A2C.A.C.A.2250.EUR.N', start='2005-01', end='2025-12')
df.head()

,KEY,FREQ,REF_AREA,BS_REP_SECTOR,BS_ITEM,MATURITY_NOT_IRATE,DATA_TYPE_MIR,AMOUNT_CAT,BS_COUNT_SECTOR,CURRENCY_TRANS,...,UNIT_INDEX_BASE,COMPILATION,COVERAGE,DECIMALS,NAT_TITLE,SOURCE_AGENCY,TITLE,TITLE_COMPL,UNIT,UNIT_MULT
0,MIR.M.ES.B.A2C.A.C.A.2250.EUR.N,M,ES,B,A2C,A,C,A,2250,EUR,...,NaN,NaN,NaN,4,NaN,ES2,Bank interest rates - loans to households for ...,"Spain, Annual percentage rate of charge (APRC)...",PCPA,0
1,MIR.M.ES.B.A2C.A.C.A.2250.EUR.N,M,ES,B,A2C,A,C,A,2250,EUR,...,NaN,NaN,NaN,4,NaN,ES2,Bank interest rates - loans to households for ...,"Spain, Annual percentage rate of charge (APRC)...",PCPA,0
2,MIR.M.ES.B.A2C.A.C.A.2250.EUR.N,M,ES,B,A2C,A,C,A,2250,EUR,...,NaN,NaN,NaN,4,NaN,ES2,Bank interest rates - loans to households for ...,"Spain, Annual percentage rate of charge (APRC)...",PCPA,0
3,MIR.M.ES.B.A2C.A.C.A.2250.EUR.N,M,ES,B,A2C,A,C,A,2250,EUR,...,NaN,NaN,NaN,4,NaN,ES2,Bank interest rates - loans to households for ...,"Spain, Annual percentage rate of charge (APRC)...",PCPA,0
4,MIR.M.ES.B.A2C.A.C.A.2250.EUR.N,M,ES,B,A2C,A,C,A,2250,EUR,...,NaN,NaN,NaN,4,NaN,ES2,Bank interest rates - loans to households for ...,"Spain, Annual percentage rate of charge (APRC)...",PCPA,0


Nos quedamos con las columnas que nos interesan

In [103]:
df = df[['TIME_PERIOD', 'OBS_VALUE']]
print(df.head())
df.to_csv('APRC_hipotecas_nuevas_men.csv', index=False)
print('DataFrame exportado a APRC_hipotecas_nuevas.csv')

  TIME_PERIOD  OBS_VALUE
0     2005-01       3.43
1     2005-02       3.44
2     2005-03       3.42
3     2005-04       3.41
4     2005-05       3.42
DataFrame exportado a APRC_hipotecas_nuevas.csv


## Trimestralizar los datos de APRC_hipotecas_nuevas.csv
Cargamos el archivo APRC_hipotecas_nuevas.csv en un DataFrame de pandas, conviertimos la columna `TIME_PERIOD` en objetos datetime, recalculamos los datos de una frecuencia mensual a una frecuencia trimestral (‘QS’), calculamos la media de OBS_VALUE para cada trimestre y mostramos el head.

## Cargamos el CSV


In [104]:
import pandas as pd

df = pd.read_csv('APRC_hipotecas_nuevas_men.csv')
df.head()

,TIME_PERIOD,OBS_VALUE
0,2005-01,3.43
1,2005-02,3.44
2,2005-03,3.42
3,2005-04,3.41
4,2005-05,3.42


## Adaptación del formato de las fechas

Convertimos la columna 'TIME_PERIOD' a tipo fecha para poder manitular el dataframe


In [105]:
df['TIME_PERIOD'] = pd.to_datetime(df['TIME_PERIOD'], format='%Y-%m')
df = df.set_index('TIME_PERIOD')
df.head()

,OBS_VALUE
TIME_PERIOD,
2005-01-01,3.43
2005-02-01,3.44
2005-03-01,3.42
2005-04-01,3.41
2005-05-01,3.42


## Pasamos los datos a trimestrales

### Subtask:
Remuestramos el DataFrame a una frecuencia trimestral ('QS', inicio de trimestre) y calculamos la media de `OBS_VALUE` para cada trimestre. De este modo convertimos los tipos de interés mensuales en promedios trimestrales mediante una serie en pandas.


In [106]:
df_quarterly = df['OBS_VALUE'].resample('QS').mean()
print('Datos reconvertidos a trimestrales:')
df_quarterly.name = 'APRC_VALUE'
print(df_quarterly.head())

Datos reconvertidos a trimestrales:
TIME_PERIOD
2005-01-01    3.430000
2005-04-01    3.393333
2005-07-01    3.286667
2005-10-01    3.373333
2006-01-01    3.763333
Freq: QS-JAN, Name: APRC_VALUE, dtype: float64


In [107]:
df_quarterly.to_csv("APRC_hipotecas_trimestral.csv", index=True)

# Importamos el dato `FM.Q.U2.EUR.RT.MM.EURIBOR1YD_.HSTA` desde el BCE, ya que lo necesitamos para saber el euribor 12M. Usamos en nuestro modelo el euribor a 12M,ya que es el que actúa como ancla psicológica al consumidor; es el que mira antes de ir a firmar a un banco para saber si los tipos que le ofrecen son caros o baratos.

In [108]:
from ecbdata import ecbdata

df = ecbdata.get_series('FM.Q.U2.EUR.RT.MM.EURIBOR1YD_.HSTA', start='2005-01', end='2025-12')
df.head()

,KEY,FREQ,REF_AREA,CURRENCY,PROVIDER_FM,INSTRUMENT_FM,PROVIDER_FM_ID,DATA_TYPE_FM,TIME_PERIOD,OBS_VALUE,...,UNIT_INDEX_BASE,COMPILATION,COVERAGE,DECIMALS,SOURCE_AGENCY,SOURCE_PUB,TITLE,TITLE_COMPL,UNIT,UNIT_MULT
0,FM.Q.U2.EUR.RT.MM.EURIBOR1YD_.HSTA,Q,U2,EUR,RT,MM,EURIBOR1YD_,HSTA,2005-Q1,2.319097,...,NaN,NaN,NaN,4,NaN,NaN,"Euribor 1-year - Historical close, average of ...",Euro area (changing composition) - Money Marke...,PCPA,0
1,FM.Q.U2.EUR.RT.MM.EURIBOR1YD_.HSTA,Q,U2,EUR,RT,MM,EURIBOR1YD_,HSTA,2005-Q2,2.185862,...,NaN,NaN,NaN,4,NaN,NaN,"Euribor 1-year - Historical close, average of ...",Euro area (changing composition) - Money Marke...,PCPA,0
2,FM.Q.U2.EUR.RT.MM.EURIBOR1YD_.HSTA,Q,U2,EUR,RT,MM,EURIBOR1YD_,HSTA,2005-Q3,2.204288,...,NaN,NaN,NaN,4,NaN,NaN,"Euribor 1-year - Historical close, average of ...",Euro area (changing composition) - Money Marke...,PCPA,0
3,FM.Q.U2.EUR.RT.MM.EURIBOR1YD_.HSTA,Q,U2,EUR,RT,MM,EURIBOR1YD_,HSTA,2005-Q4,2.627969,...,NaN,NaN,NaN,4,NaN,NaN,"Euribor 1-year - Historical close, average of ...",Euro area (changing composition) - Money Marke...,PCPA,0
4,FM.Q.U2.EUR.RT.MM.EURIBOR1YD_.HSTA,Q,U2,EUR,RT,MM,EURIBOR1YD_,HSTA,2006-Q1,2.954492,...,NaN,NaN,NaN,4,NaN,NaN,"Euribor 1-year - Historical close, average of ...",Euro area (changing composition) - Money Marke...,PCPA,0


Renombramos la columna `obs_value` a `euribor_value` para concretar de qué dato se trata.

In [109]:
df = df[['TIME_PERIOD', 'OBS_VALUE']]
df.rename(columns={'OBS_VALUE': 'EURIBOR_VALUE'}, inplace=True)
print(df.head())

  TIME_PERIOD  EURIBOR_VALUE
0     2005-Q1       2.319097
1     2005-Q2       2.185862
2     2005-Q3       2.204288
3     2005-Q4       2.627969
4     2006-Q1       2.954492


Convertimos las fechas a formato fecha

In [110]:
df['TIME_PERIOD'] = pd.to_datetime(df['TIME_PERIOD'])
df = df.set_index('TIME_PERIOD')
print('DataFrame euribor con fechas en formato fecha:')
print(df.head())

DataFrame euribor con fechas en formato fecha:
             EURIBOR_VALUE
TIME_PERIOD               
2005-01-01        2.319097
2005-04-01        2.185862
2005-07-01        2.204288
2005-10-01        2.627969
2006-01-01        2.954492


/var/folders/pq/jvlxhvnn4xb2whw_2kjk6tv40000gn/T/ipykernel_740/3739840634.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['TIME_PERIOD'] = pd.to_datetime(df['TIME_PERIOD'])


# La distancia entre APRC y euribor capta el spread bancario de forma simplificada, es decir, capta la curva de tipos. Hagamos la diferencia entre ambos y generemos un nuevo csv con la diferencia de tipos

## Combinamos ambas series `euribor` y `APRC`



Ahora que tanto los datos del APRC como los del Euríbor están preparados con un índice de fechas común, el siguiente paso lógico es combinar estos dos conjuntos de datos para calcular el spread bancario (curva de tipos).



In [111]:
df_combined = pd.merge(df_quarterly, df, left_index=True, right_index=True, how='inner')
df_combined['Diferencia'] = df_combined['APRC_VALUE'] - df_combined['EURIBOR_VALUE']
print('DataFrame combinado con diferencial calculado')
print(df_combined.head())
df_combined.to_csv('tipos_combinados_hipotecas.csv')

DataFrame combinado con diferencial calculado
             APRC_VALUE  EURIBOR_VALUE  Diferencia
TIME_PERIOD                                       
2005-01-01     3.430000       2.319097    1.110903
2005-04-01     3.393333       2.185862    1.207472
2005-07-01     3.286667       2.204288    1.082379
2005-10-01     3.373333       2.627969    0.745365
2006-01-01     3.763333       2.954492    0.808841


**Reasoning**:
El *banking spread* ha sido calculado y almacenado en `df_combined`. El paso final de la tarea principal es exportar estos datos a un archivo CSV llamado `spread_hipotecas_euribor.csv`, asegurándonos de que contenga las columnas ‘Fecha’ y ‘Diferencia’. Para lograr esto, necesito resetear el índice de `df_combined` y renombrar la columna `TIME_PERIOD` a ‘Fecha’ antes de exportar.



In [ ]:
df_final = df_combined.reset_index()
df_final.rename(columns={'TIME_PERIOD': 'Fecha'}, inplace=True)
df_final = df_final[['Fecha', 'Diferencia']]
df_final.to_csv('spread_hipotecas.csv', index=False)
print('Datos del spread exportados a spread_hipotecas_euribor.csv:')
print(df_final.head())

Datos del spread exportados a spread_hipotecas_euribor.csv:
       Fecha  Diferencia
0 2005-01-01    1.110903
1 2005-04-01    1.207472
2 2005-07-01    1.082379
3 2005-10-01    0.745365
4 2006-01-01    0.808841


# Limpiamos nuestros CSV

In [130]:
df = pd.read_csv('edad_media_provincias_an.csv', sep=';', encoding='latin-1')
df.head()

,Provincias,Sexo,Periodo,Total
0,Albacete,Ambos sexos,2025,"44,83"
1,Albacete,Ambos sexos,2024,"44,59"
2,Albacete,Ambos sexos,2023,"44,34"
3,Albacete,Ambos sexos,2022,"44,14"
4,Albacete,Ambos sexos,2021,"43,85"


In [133]:
df = pd.read_excel("pib_pc_prov_an.xlsx")
df

,Provincia,Año,PIB_per_capita_euros
0,Albacete,2005,16415
1,Albacete,2006,17444
2,Albacete,2007,18122
3,Albacete,2008,18955
4,Albacete,2009,18798
...,...,...,...
1035,Ávila,2020,19336
1036,Ávila,2021,20409
1037,Ávila,2022,21756
1038,Ávila,2023,23275


INE suele tener el código de la provincia a la izquierda, lo eliminamos para evitar problemas a la unica de unir CSV en un único dataset.

In [128]:
import pandas as pd

archivos = [
    "edad_media_provincias.csv",
    "hipotecas_por_prov.csv",
    "ipc_ccaa.csv",
    "poblacion_por_prov_combinado.csv",
    "transmision_propiedades_prov.csv",
]

columnas_provincia = ["Provincias", "provincia", "Provincia", "PROVINCIA","Comunidades Autonomas","Comunidades y Ciudades Autónomas"]

for archivo in archivos:
    df = pd.read_csv(archivo, sep=";", encoding="latin1")
    
    col = next((c for c in columnas_provincia if c in df.columns), None)
    
    if col:
        df[col] = df[col].str.replace(r"^\d+\s+", "", regex=True)
        df.to_csv(archivo, sep=";", index=False, encoding="utf-8")
        print(f"{archivo} — columna '{col}' limpiada")
    else:
        print(f"{archivo} — no se encontró columna de provincia")

edad_media_provincias.csv — columna 'Provincias' limpiada
hipotecas_por_prov.csv — columna 'Provincias' limpiada
ipc_ccaa.csv — no se encontró columna de provincia
poblacion_por_prov_combinado.csv — columna 'Provincias' limpiada
transmision_propiedades_prov.csv — columna 'Provincias' limpiada


---

# * Fusión de datos de Población Extranjera por Provincias (2023-2025)

Los datos del INE sobre población extranjera por provincias solo cubren hasta el año **2022**. Para extender la serie hasta **2025**, recurrimos a los datos del **Ministerio de Inclusión, Seguridad Social y Migraciones**, disponibles en su [Observatorio Permanente de la Inmigración](https://expinterweb.inclusion.gob.es/jaxiPx/Tabla.htm?path=/Stock/l0/&file=TD_PV_GE_PNDAD_NTO.px&L=0).

Esta fuente publica los datos con **granularidad trimestral**, por lo que debemos **anualizarlos** (promedio de los cuatro trimestres) antes de concatenarlos con la serie del INE.

Se utilizan dos ficheros CSV:

| Fichero | Fuente | Cobertura | Granularidad |
|---|---|---|---|
| `pob_ext_2005_2022.csv` | INE — Padrón Municipal | 2005–2022 | Anual |
| `pob_ext_2023_2025.csv` | Ministerio de Inclusión | 2023–2025 | **Trimestral** |

El resultado se exporta a `pob_ext_provincias_2005_2025.csv` con columnas `año`, `provincia` y `pob_ext`.

In [ ]:
import pandas as pd
import numpy as np

---
** Paso 1 — Carga y limpieza del fichero anual (2005-2022)

El fichero `pob_ext_2005_2022.csv` usa `;` como separador y codificación **latin-1**.  
Contiene las columnas: `Nacionalidad`, `Provincias`, `Sexo`, `Periodo`, `Total`.

Nos interesan únicamente las filas donde:
- `Sexo == 'Ambos sexos'` → evitamos duplicar conteos por género.
- `Nacionalidad == 'TOTAL EXTRANJEROS'` → descartamos desgloses por país de origen.

La columna `Provincias` viene con el código numérico delante (ej. `"02 Albacete"`), así que lo eliminamos para quedarnos sólo con el nombre.  
La columna `Total` usa `.` como separador de miles y `,` como decimal (formato español), por lo que la convertimos a numérico.

In [ ]:
df_anual = pd.read_csv(
    'pob_ext_2005_2022.csv',
    sep=';',
    encoding='latin-1',
    dtype=str           # cargamos todo como texto para limpiar primero
)

print('Shape original:', df_anual.shape)
print('Columnas:', df_anual.columns.tolist())
df_anual.head()

In [ ]:
# ── Filtrado: solo total de ambos sexos y total extranjeros ──────────────────
df_anual = df_anual[
    (df_anual['Sexo'].str.strip() == 'Ambos sexos') &
    (df_anual['Nacionalidad'].str.strip() == 'TOTAL EXTRANJEROS')
].copy()

# ── Limpieza de la columna Provincias: eliminar código numérico inicial ──────
# Ejemplo: "02 Albacete" → "Albacete"
df_anual['provincia'] = (
    df_anual['Provincias']
    .str.strip()
    .str.replace(r'^\d+\s+', '', regex=True)
)

# ── Conversión de Total a numérico (formato español: "27.558" → 27558) ───────
df_anual['pob_ext'] = (
    df_anual['Total']
    .str.strip()
    .str.replace('.', '', regex=False)   # quitar separador de miles
    .str.replace(',', '.', regex=False)  # coma decimal → punto
    .astype(float)
    .astype(int)
)

# ── Selección y renombrado de columnas relevantes ────────────────────────────
df_anual = df_anual[['Periodo', 'provincia', 'pob_ext']].copy()
df_anual.rename(columns={'Periodo': 'año'}, inplace=True)
df_anual['año'] = df_anual['año'].astype(int)

print('Filas tras filtrado:', len(df_anual))
print('Años disponibles:', sorted(df_anual['año'].unique()))
df_anual.head(10)

---
** Paso 2 — Carga y anualización del fichero trimestral (2023-2025)

El fichero `pob_ext_2023_2025.csv` también usa `;` como separador y codificación **latin-1**.  
Las columnas relevantes son `Provincia`, `Fecha` (formato `YYYY-MM-DD`) y `Total`.

Para obtener el **valor anual**, calculamos la **media aritmética de los cuatro trimestres** de cada año.  
Esto es equivalente a:

$$\text{pob}_{\text{anual}} = \frac{\text{T1} + \text{T2} + \text{T3} + \text{T4}}{4}$$

> **Nota sobre años incompletos:** si un año tiene menos de 4 trimestres (ej. 2025), el promedio se sigue calculando pero con los trimestres disponibles. Esto se indica en el DataFrame resultante.

Filtros aplicados:
- `Principales nacionalidades == 'Todas las nacionalidades'` → total de extranjeros.
- `Grupo de edad == 'Total'` → todos los grupos de edad.
- `Lugar de nacimiento == 'Total'` → sin desagregación por lugar de nacimiento.
- `Tipo de documentación == 'Total'` → sin desagregación por documento.

In [ ]:
df_trim = pd.read_csv(
    'pob_ext_2023_2025.csv',
    sep=';',
    encoding='latin-1',
    dtype=str
)

print('Shape original:', df_trim.shape)
print('Columnas:', df_trim.columns.tolist())
df_trim.head()

In [ ]:
# ── Filtrado: exactamente las mismas categorías agregadas ────────────────────
filtro = (
    (df_trim['Principales nacionalidades'].str.strip() == 'Todas las nacionalidades') &
    (df_trim['Grupo de edad'].str.strip() == 'Total') &
    (df_trim['Lugar de nacimiento'].str.strip() == 'Total') &
    (df_trim['Tipo de documentación'].str.strip() == 'Total')
)
df_trim = df_trim[filtro].copy()

print('Filas tras filtrado:', len(df_trim))

# ── Extraer año desde la columna Fecha (YYYY-MM-DD) ─────────────────────────
df_trim['año'] = pd.to_datetime(df_trim['Fecha'].str.strip()).dt.year

# ── Conversión de Total a numérico ───────────────────────────────────────────
df_trim['pob_ext'] = (
    df_trim['Total']
    .str.strip()
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

df_trim['provincia'] = df_trim['Provincia'].str.strip()

# ── Comprobamos cuántos trimestres hay por año/provincia ─────────────────────
conteo = df_trim.groupby(['año', 'provincia']).size().reset_index(name='n_trimestres')
print('\nTrimestres por año (valores distintos):')
print(conteo['n_trimestres'].value_counts().sort_index())

In [ ]:
# ── Anualización: promedio de los trimestres disponibles ────────────────────
df_anual_trim = (
    df_trim
    .groupby(['año', 'provincia'], as_index=False)
    .agg(
        pob_ext=('pob_ext', 'mean'),
        n_trimestres=('pob_ext', 'count')
    )
)

# Redondeamos al entero más cercano para coherencia con el fichero anual
df_anual_trim['pob_ext'] = df_anual_trim['pob_ext'].round(0).astype(int)

# Avisamos si algún año tiene datos incompletos
incompletos = df_anual_trim[df_anual_trim['n_trimestres'] < 4]
if not incompletos.empty:
    print('⚠️  Años con menos de 4 trimestres (promedio parcial):')
    print(incompletos[['año', 'provincia', 'n_trimestres']].drop_duplicates('año'))
else:
    print('✅ Todos los años tienen 4 trimestres completos.')

df_anual_trim = df_anual_trim[['año', 'provincia', 'pob_ext']]
print('\nShape del dataframe anualizado:', df_anual_trim.shape)
df_anual_trim.head(10)

---
** Paso 3 — Homogenización de nombres de provincia

Los dos ficheros provienen de fuentes distintas del INE y usan criterios de nomenclatura diferentes:

- El fichero **2005-2022** usa el nombre oficial en castellano (ej. `Álava`).
- El fichero **2023-2025** usa a veces el nombre en la lengua co-oficial (ej. `Araba/Álava`).

Normalizamos los nombres del segundo fichero al formato del primero mediante un diccionario de mapeo.  
Al final verificamos que todas las provincias del fichero trimestral tengan correspondencia en el anual.

In [ ]:
# ── Provincias únicas en cada fuente ────────────────────────────────────────
provs_anual = set(df_anual['provincia'].unique())
provs_trim  = set(df_anual_trim['provincia'].unique())

print('Provincias SOLO en fichero anual (no están en trimestral):')
print(sorted(provs_anual - provs_trim))

print('\nProvincias SOLO en fichero trimestral (necesitan mapeo):')
print(sorted(provs_trim - provs_anual))

In [ ]:
# ── Diccionario de mapeo: nombre en trimestral → nombre estándar ─────────────
mapeo_provincias = {
    'Araba/Álava'       : 'Álava',
    'Bizkaia'           : 'Vizcaya',
    'Gipuzkoa'          : 'Guipúzcoa',
    'Girona'            : 'Gerona',
    'Lleida'            : 'Lérida',
    'Tarragona'         : 'Tarragona',    # suele coincidir, incluida por completitud
    'A Coruña'          : 'La Coruña',
    'Ourense'           : 'Orense',
    'Balears, Illes'    : 'Balears, Illes',   # puede coincidir
    'Rioja, La'         : 'Rioja, La',
    'Palmas, Las'       : 'Palmas, Las',
    # Añade aquí más mapeos si el diagnóstico anterior los detectó
}

df_anual_trim['provincia'] = df_anual_trim['provincia'].replace(mapeo_provincias)

# ── Verificación post-mapeo ─────────────────────────────────────────────────
provs_trim_post = set(df_anual_trim['provincia'].unique())
sin_mapear = provs_trim_post - provs_anual

if sin_mapear:
    print('⚠️  Aún hay provincias del trimestral sin equivalente en el anual:')
    print(sorted(sin_mapear))
else:
    print('✅ Todos los nombres de provincia están homogeneizados.')

---
** Paso 4 — Fusión y exportación del fichero final

Concatenamos verticalmente los dos DataFrames ya limpios y homogeneizados.

- El fichero anual aporta los años **2005 a 2022**.
- El fichero trimestral anualizado aporta los años **2023 a 2025** (o los disponibles).

No debe haber solapamiento de años entre ambas fuentes.  
Ordenamos el resultado por `provincia` y `año` y lo exportamos a `pob_ext_provincias_2005_2025.csv`.

In [ ]:
# ── Comprobación de solapamiento de años ─────────────────────────────────────
años_anual = set(df_anual['año'].unique())
años_trim  = set(df_anual_trim['año'].unique())
solapamiento = años_anual & años_trim

if solapamiento:
    print(f'⚠️  Hay solapamiento en los años: {sorted(solapamiento)}')
    print('   Se eliminarán esos años del fichero anual para evitar duplicados.')
    df_anual = df_anual[~df_anual['año'].isin(solapamiento)]
else:
    print('✅ No hay solapamiento de años. La concatenación es limpia.')

print(f'\nAños en fichero anual:      {sorted(df_anual["año"].unique())}')
print(f'Años en fichero trimestral: {sorted(df_anual_trim["año"].unique())}')

In [ ]:
# ── Concatenación vertical ───────────────────────────────────────────────────
df_final = pd.concat([df_anual, df_anual_trim], ignore_index=True)

# ── Ordenación por provincia (alfabética) y año ──────────────────────────────
df_final = df_final.sort_values(['provincia', 'año']).reset_index(drop=True)

print('Shape del DataFrame final:', df_final.shape)
print('Años cubiertos:', sorted(df_final['año'].unique()))
print('Número de provincias:', df_final['provincia'].nunique())
df_final.head(15)

In [ ]:
# ── Estadísticas descriptivas básicas ───────────────────────────────────────
df_final.describe()

In [ ]:
# ── Comprobación de valores nulos ────────────────────────────────────────────
nulos = df_final.isnull().sum()
print('Valores nulos por columna:')
print(nulos)

# ── Comprobación de duplicados ────────────────────────────────────────────────
dupl = df_final.duplicated(subset=['año', 'provincia']).sum()
print(f'\nDuplicados (año + provincia): {dupl}')

In [ ]:
# ── Exportación del fichero final ─────────────────────────────────────────────
output_path = 'pob_ext_provincias_2005_2025.csv'

df_final.to_csv(
    output_path,
    index=False,
    sep=';',
    encoding='utf-8-sig'   # utf-8 con BOM para compatibilidad con Excel
)

print(f'✅ Fichero exportado correctamente: {output_path}')
print(f'   Registros totales: {len(df_final):,}')
print(f'   Provincias:        {df_final["provincia"].nunique()}')
print(f'   Rango temporal:    {df_final["año"].min()} – {df_final["año"].max()}')

---
** Resumen del proceso

| Paso | Acción | Resultado |
|------|--------|-----------|
| 1 | Carga y limpieza de `pob_ext_2005_2022.csv` | DataFrame anual 2005-2022 |
| 2 | Carga de `pob_ext_2023_2025.csv` y **anualización por media trimestral** | DataFrame anual 2023-2025 |
| 3 | Homogenización de nombres de provincia (mapeo lenguas co-oficiales) | Nombres unificados |
| 4 | Concatenación, validación y exportación | `pob_ext_provincias_2005_2025.csv` |

La serie resultante cubre **52 provincias** españolas durante **21 años** (2005-2025), con una única observación anual por provincia y un único indicador: `pob_ext` (población extranjera empadronada).

---

# * Fusión de datos de Transmisión de Propiedades por Provincias (2005–2025)

Los datos del INE sobre transmisión de propiedades (compraventas) por provincias tienen **cobertura mensual desde 2007**, pero carecen de datos para **2005 y 2006**.
Para completar la serie histórica utilizamos un fichero adicional del propio INE con los datos trimestrales de esos dos años:

| Fichero | Cobertura | Granularidad |
|---|---|---|
| `transmisiones propiedades 2005-2006.xls` | 2005–2006 | **Trimestral** (formato tabla ancha) |
| `transmision_propiedades_prov.csv` | 2007–2025 | **Mensual** → agregado a trimestral |

El CSV principal presenta además un problema de **triple codificación UTF-8** (mojibake triple) que debe corregirse antes de procesarlo.

El resultado se exporta a `transmision_prov_trim_tot.csv` con columnas `año`, `trimestre`, `provincia` y `transmisiones`.

---

** Paso 1 — Importar librerías y corregir el encoding del CSV

El fichero `transmision_propiedades_prov.csv` fue guardado con un **triple error de codificación** (los bytes UTF-8 fueron interpretados como latin-1 tres veces seguidas). Para recuperar el texto original hay que invertir ese proceso: leer los bytes en crudo y aplicar cuatro rondas de decodificación/codificación alternando UTF-8 y latin-1.

In [ ]:
import pandas as pd
import io

# Leer el fichero en crudo (bytes) y reparar el triple mojibake
with open('transmision_propiedades_prov.csv', 'rb') as f:
    raw = f.read()

# Cada ronda invierte una capa de codificación errónea:
# utf-8 → latin-1 → utf-8 → latin-1 → utf-8 → latin-1 → utf-8
texto_limpio = (
    raw
    .decode('utf-8').encode('latin-1')
    .decode('utf-8').encode('latin-1')
    .decode('utf-8').encode('latin-1')
    .decode('utf-8')
)

df_csv = pd.read_csv(io.StringIO(texto_limpio), sep=';', dtype=str)
print('Shape CSV:', df_csv.shape)
print('Columnas:', df_csv.columns.tolist())
df_csv.head()

---

** Paso 2 — Filtrar y convertir datos del CSV (2007-2025) de mensual a trimestral

El CSV contiene datos **mensuales** con columna `Periodo` en formato `YYYYMNN` (ej. `2025M11`).  
Filtramos únicamente las filas donde `Naturaleza de la finca == 'Total'` y convertimos a trimestral **sumando los tres meses** de cada trimestre por provincia.

La correspondencia mes → trimestre es:

| Meses | Trimestre |
|---|---|
| 01, 02, 03 | T1 |
| 04, 05, 06 | T2 |
| 07, 08, 09 | T3 |
| 10, 11, 12 | T4 |

In [ ]:
# ── Filtrar solo filas de total (sin desagregación por tipo de finca) ──────
df_csv = df_csv[df_csv['Naturaleza de la finca'].str.strip() == 'Total'].copy()

# ── Extraer año y calcular trimestre desde el código de periodo ─────────────
df_csv['año']       = df_csv['Periodo'].str[:4].astype(int)
df_csv['mes']       = df_csv['Periodo'].str[5:].astype(int)
df_csv['trimestre'] = ((df_csv['mes'] - 1) // 3 + 1)

# ── Convertir Total a numérico (formato español: '5.037' → 5037) ───────────
df_csv['transmisiones'] = (
    df_csv['Total'].str.strip()
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float).astype(int)
)
df_csv['provincia'] = df_csv['Provincias'].str.strip()

# ── Agregar a trimestral: suma de los meses del trimestre ──────────────────
df_csv_trim = (
    df_csv[df_csv['provincia'] != 'Total']
    .groupby(['año', 'trimestre', 'provincia'], as_index=False)
    ['transmisiones'].sum()
)

print('Shape trimestral CSV:', df_csv_trim.shape)
print('Años cubiertos:', sorted(df_csv_trim['año'].unique()))
df_csv_trim.head()

---

** Paso 3 — Leer y transformar el XLS de 2005-2006 (formato ancho → largo)

El fichero XLS tiene un formato **tabla ancha** con 8 columnas de datos (4 trimestres × 2 años).  
Los datos empiezan en la fila 14 (tras varias filas de cabecera y título).  
Se descartan las filas de **totales nacionales y comunidades autónomas** para quedarse solo con provincias.

Luego se hace un **melt** (pivot de tabla ancha a tabla larga) para obtener el mismo formato que el CSV.

In [ ]:
# ── Leer el XLS ignorando las filas de cabecera ─────────────────────────────
xls_raw = pd.read_excel('transmisiones propiedades 2005-2006.xls', header=None)

# Datos desde fila 14: col 1 = provincia, cols 2-5 = 2005 T1-T4, cols 6-9 = 2006 T1-T4
datos = xls_raw.iloc[14:, 1:10].copy()
datos.columns = ['provincia',
                 '2005T1','2005T2','2005T3','2005T4',
                 '2006T1','2006T2','2006T3','2006T4']
datos = datos.dropna(subset=['provincia'])
datos['provincia'] = datos['provincia'].astype(str).str.strip()

# ── Convertir columnas de datos a numérico ──────────────────────────────────
trim_cols = ['2005T1','2005T2','2005T3','2005T4','2006T1','2006T2','2006T3','2006T4']
for c in trim_cols:
    datos[c] = pd.to_numeric(
        datos[c].astype(str).str.replace('.', '', regex=False).str.strip(),
        errors='coerce'
    )

# ── Eliminar filas de totales y comunidades autónomas ───────────────────────
excluir = [
    'TOTAL NACIONAL','Andalucía','Aragón','Asturias (Principado de)',
    'Balears (Illes)','Canarias','Cantabria','Castilla y León',
    'Castilla-La Mancha','Cataluña','Comunitat Valenciana','Extremadura',
    'Galicia','Madrid (Comunidad de)','Murcia (Región de)',
    'Navarra (Comunidad Foral de)','País Vasco','Rioja (La)','Ceuta y Melilla'
]
datos = datos[~datos['provincia'].isin(excluir)].dropna()

# ── Melt: formato ancho → largo ─────────────────────────────────────────────
df_xls_long = datos.melt(id_vars='provincia', var_name='periodo_raw', value_name='transmisiones')
df_xls_long['año']       = df_xls_long['periodo_raw'].str[:4].astype(int)
df_xls_long['trimestre'] = df_xls_long['periodo_raw'].str[5].astype(int)
df_xls_long['transmisiones'] = df_xls_long['transmisiones'].astype(int)
df_xls_long = df_xls_long[['año', 'trimestre', 'provincia', 'transmisiones']]

print('Shape XLS largo:', df_xls_long.shape)
print('Provincias:', df_xls_long['provincia'].nunique())
df_xls_long.head()

---

** Paso 4 — Homogenización de nombres de provincia

El XLS utiliza algunas variantes de nombre distintas al CSV.  
Se aplica un diccionario de mapeo para unificar al formato del CSV:

| Nombre en XLS | Nombre estándar (CSV) |
|---|---|
| `Araba/Alava` | `Araba/Álava` |
| `Coruña (A)` | `Coruña, A` |
| `Palmas (Las)` | `Palmas, Las` |

In [ ]:
mapeo_provincias = {
    'Araba/Alava'  : 'Araba/Álava',
    'Coruña (A)'   : 'Coruña, A',
    'Palmas (Las)' : 'Palmas, Las',
}
df_xls_long['provincia'] = df_xls_long['provincia'].replace(mapeo_provincias)

# ── Verificación ─────────────────────────────────────────────────────────────
provs_csv = set(df_csv_trim['provincia'].unique())
provs_xls = set(df_xls_long['provincia'].unique())
sin_mapear = provs_xls - provs_csv
if sin_mapear:
    print('⚠️  Sin equivalente en CSV:', sorted(sin_mapear))
else:
    print('✅ Todos los nombres de provincia están homogeneizados.')

---

** Paso 5 — Fusión, validación y exportación

Concatenamos verticalmente los dos DataFrames (XLS 2005-2006 + CSV 2007-2025), verificamos que no hay solapamiento de años ni duplicados, y exportamos el resultado.

In [ ]:
# ── Verificar solapamiento de años ──────────────────────────────────────────
años_xls = set(df_xls_long['año'].unique())
años_csv = set(df_csv_trim['año'].unique())
solap = años_xls & años_csv
if solap:
    print(f'⚠️  Solapamiento en años: {sorted(solap)} → se elimina del CSV')
    df_csv_trim = df_csv_trim[~df_csv_trim['año'].isin(solap)]
else:
    print('✅ Sin solapamiento de años.')

# ── Concatenación y ordenación ──────────────────────────────────────────────
df_final = pd.concat([df_xls_long, df_csv_trim], ignore_index=True)
df_final = df_final.sort_values(['provincia', 'año', 'trimestre']).reset_index(drop=True)

print('Shape final:', df_final.shape)
print('Años cubiertos:', sorted(df_final['año'].unique()))
print('Provincias:', df_final['provincia'].nunique())

# ── Validación ───────────────────────────────────────────────────────────────
print('\nNulos por columna:')
print(df_final.isnull().sum())
print('Duplicados (año + trimestre + provincia):', df_final.duplicated(subset=['año','trimestre','provincia']).sum())

df_final.head(12)

In [ ]:
# ── Estadísticas descriptivas ────────────────────────────────────────────────
df_final.describe()

In [ ]:
# ── Exportación ──────────────────────────────────────────────────────────────
output_path = 'transmision_prov_trim_tot.csv'

df_final.to_csv(
    output_path,
    index=False,
    sep=';',
    encoding='utf-8-sig'   # BOM para compatibilidad con Excel
)

print(f'✅ Fichero exportado: {output_path}')
print(f'   Registros totales: {len(df_final):,}')
print(f'   Provincias:        {df_final["provincia"].nunique()}')
print(f'   Rango temporal:    {df_final["año"].min()} T{df_final.loc[df_final["año"]==df_final["año"].min(),"trimestre"].min()} — {df_final["año"].max()} T{df_final.loc[df_final["año"]==df_final["año"].max(),"trimestre"].max()}')

---

** Resumen del proceso

| Paso | Acción | Resultado |
|------|--------|-----------|
| 1 | Reparación de triple mojibake UTF-8 del CSV | Texto legible |
| 2 | Filtrado y agregación mensual → trimestral (suma) | DataFrame 2007-2025 |
| 3 | Lectura del XLS y melt (ancho → largo) | DataFrame 2005-2006 |
| 4 | Homogenización de nombres de provincia | Nombres unificados |
| 5 | Fusión, validación y exportación | `transmision_prov_trim_tot.csv` |

El dataset final cubre **52 provincias** durante **21 años** (2005-2025) con granularidad **trimestral**.